# 🔴 CityFlow AI — Lab 1: Data Poisoning & Defense

**Nodo afectado:** `NODE-1 · Sensor Data (Edge)`  
**Amenaza:** Envenenamiento de datos de entrenamiento  
**Dataset:** Metro Interstate Traffic Volume — UCI ML Repository (ID 492)

---

Este notebook se conecta al **pipeline real** que estás viendo en los logs del frontend.  
Sigue la guía del panel lateral paso a paso. Cada celda = un paso.

> ⚠️ Entorno controlado. El ataque forma parte de un laboratorio local controlado.


In [ ]:
%matplotlib inline

import requests
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# URL del backend real dentro de la red Docker.
# El servicio del backend se llama "backend" en docker-compose.
BACKEND = "http://backend:8000"


def call_pipeline(mode="vulnerable", n_readings=10):
    """
    Llama al endpoint real del pipeline.

    El backend actual ejecuta Scenario 1 en modo vulnerable y devuelve:
      - metrics
      - data.n1  -> sensor-data
      - data.n2  -> edge-preprocessing
      - data.n3  -> traffic-inference / actuator
      - data.n4  -> trainer / retraining

    Devuelve:
      (metrics, n1, n2, n3, n4)
    """

    urls = [
        f"{BACKEND}/api/scenarios/1/run",
        f"{BACKEND}/scenarios/1/run",
    ]

    last_error = None

    for url in urls:
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
            result = resp.json()

            data = result.get("data", {})

            metrics = result.get("metrics", {})
            n1 = data.get("n1", {})
            n2 = data.get("n2", {})
            n3 = data.get("n3", {})
            n4 = data.get("n4", {})

            return metrics, n1, n2, n3, n4

        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"No se pudo conectar con el backend real: {last_error}")


print("✓ Helper listo.")
print("  Backend:", BACKEND)
print("  Usa call_pipeline() para llamar al pipeline real.")


---
## Paso 1 — Deconstruye el exploit

### ¿Qué datos entran normalmente por NODE-1?

Llamamos al pipeline real y observamos qué recibe `NODE-1 · Sensor Data`.

Este nodo toma lecturas del dataset UCI y, en modo vulnerable, reenvía incluso lecturas físicamente imposibles.


In [ ]:
metrics, n1, n2, n3, n4 = call_pipeline()

readings = n1.get("readings", [])
dropped = n1.get("dropped", [])
n1_logs = n1.get("log", [])

print("=" * 72)
print("  NODE-1 · Sensor Data — salida del pipeline real")
print("=" * 72)
print(f"  Modo del nodo:                 {n1.get('mode', 'unknown')}")
print(f"  Lecturas aceptadas:            {len(readings)}")
print(f"  Lecturas rechazadas:           {len(dropped)}")
print(f"  Lecturas envenenadas detectadas en metrics: {metrics.get('poisoned_readings', 0)}")
print()

print("  Logs NODE-1:")
for line in n1_logs[:10]:
    marker = "🔴" if "-5000" in line or "_poisoned" in line.lower() else "  "
    print(f"  {marker} {line}")

print()

vols = [
    r.get("traffic_volume", 0)
    for r in readings
    if isinstance(r, dict)
]

if vols:
    print("  traffic_volume en este batch:")
    print(f"    min   = {min(vols)}")
    print(f"    max   = {max(vols)}")
    print(f"    media = {sum(vols) / len(vols):.0f}")
    print()
    print("  ⚠️ Observación:")
    print("     Una carretera no puede tener un número negativo de coches.")
    print("     Si NODE-1 acepta traffic_volume < 0, falta validación de entrada.")


In [ ]:
# Visualizar los valores de traffic_volume e identificar el valor envenenado.

vols = [
    r.get("traffic_volume", 0)
    for r in readings
    if isinstance(r, dict)
]

if not vols:
    raise RuntimeError("No hay lecturas en NODE-1. Revisa la respuesta del backend.")

ids = [f"#{i}" for i in range(len(vols))]
cols = ["#ef4444" if v < 0 else "#3b82f6" for v in vols]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    "NODE-1 · traffic_volume por lectura",
    fontsize=12,
    fontweight="bold"
)

# Bar chart
axes[0].bar(ids, vols, color=cols)
axes[0].axhline(0, color="black", linewidth=1.2, linestyle="--")
axes[0].set_xlabel("Lectura")
axes[0].set_ylabel("traffic_volume")
axes[0].set_title("Rojo = valor físicamente imposible")

if min(vols) < 0:
    poison_idx = vols.index(min(vols))
    axes[0].annotate(
        f"ATAQUE\n{min(vols)}",
        xy=(poison_idx, min(vols)),
        xytext=(poison_idx + 0.5, min(vols) * 0.6),
        arrowprops=dict(arrowstyle="->", color="red"),
        color="red",
        fontweight="bold",
        fontsize=9,
    )

# Distribution
legitimate = [v for v in vols if v >= 0]

if legitimate:
    axes[1].hist(
        legitimate,
        bins=8,
        color="#3b82f6",
        alpha=0.8,
        edgecolor="white",
        label="Lecturas legítimas",
    )

if min(vols) < 0:
    axes[1].axvline(
        min(vols),
        color="red",
        linewidth=2,
        linestyle="--",
        label=f"Envenenada ({min(vols)})",
    )

axes[1].set_xlabel("traffic_volume")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución del batch")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/node1_readings.png", dpi=120, bbox_inches="tight")
plt.show()

print()
print("✓ El valor envenenado es físicamente imposible.")
print("  El sistema lo acepta porque NODE-1 está ejecutándose en modo vulnerable.")
print()
print("→ Responde en la guía: ¿qué control de seguridad falta en NODE-1?")


---
## Paso 2 — Observa la propagación del ataque 🔴

### El dato envenenado se propaga por el pipeline

Ahora analizamos cómo el `traffic_volume = -5000` pasa de `NODE-1` a `NODE-2`, genera un `congestion_score` anómalo y llega al nodo de inferencia/decisión.


In [ ]:
metrics, n1, n2, n3, n4 = call_pipeline()

features = n2.get("features", [])
n2_logs = n2.get("log", [])
predictions = n3.get("predictions", [])
actions = n3.get("actions", [])
aggregate = n3.get("aggregate", {})

print("=" * 72)
print("  🔴 PROPAGACIÓN DEL ATAQUE POR EL PIPELINE")
print("=" * 72)
print()

print("  NODE-1 · Sensor Data")
print(f"    lecturas aceptadas:  {metrics.get('readings_received', '?')}")
print(f"    envenenadas:         {metrics.get('poisoned_readings', '?')}")
print()

print("  NODE-2 · Edge Pre-processing")
print(f"    features generadas:  {metrics.get('features_generated', '?')}")
print(f"    features anómalas:   {metrics.get('anomalous_features', '?')}")
print()

anomalous = [
    f for f in features
    if isinstance(f, dict)
    and isinstance(f.get("congestion_score"), (int, float))
    and float(f["congestion_score"]) < 0
]

normal = [
    f for f in features
    if isinstance(f, dict)
    and isinstance(f.get("congestion_score"), (int, float))
    and 0 <= float(f["congestion_score"]) <= 1
]

if anomalous:
    s = anomalous[0]
    print("  ⚠️ congestion_score ANÓMALO detectado en NODE-2:")
    print(f"     reading_id      = {s.get('reading_id')}")
    print(f"     traffic_volume  = {s.get('traffic_volume')}")
    print(f"     congestion_score= {s.get('congestion_score')}")
    print(f"     label_hint      = {s.get('label_hint')}")
    print()

print("  NODE-3 · Traffic Inference / Actuator")
print(f"    predicciones generadas: {len(predictions)}")
print(f"    estado dominante:       {aggregate.get('dominant_state', '?')}")
print(f"    avg_congestion_score:   {aggregate.get('avg_congestion_score', '?')}")
print(f"    integrity_ok:           {metrics.get('integrity_ok', '?')}")
print(f"    acciones generadas:     {len(actions)}")
print()

print("  NODE-4 · Trainer / Retraining")
print(f"    drift_score:            {metrics.get('drift_score', 0):.3f}")
print(f"    retraining activado:    {n4.get('retrain_triggered', '?')}")
print()

print("  Logs NODE-2:")
for line in n2_logs[:10]:
    marker = "🔴" if any(x in line for x in ["-0.", "ANOMAL", "SKIP", "unknown"]) else "  "
    print(f"  {marker} {line}")

print()
print("  Acciones generadas por NODE-3:")
if actions:
    for action in actions[:10]:
        print("   →", action)
else:
    print("   No se han devuelto acciones en esta ejecución.")


In [ ]:
# Visualizar el impacto del ataque en los nodos principales.

scores = [
    f.get("congestion_score", 0)
    for f in features
    if isinstance(f, dict)
]

score_colors = ["#ef4444" if s < 0 else "#3b82f6" for s in scores]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(
    "Impacto del Data Poisoning en el pipeline",
    fontsize=13,
    fontweight="bold",
    color="red",
)

# NODE-1: traffic_volume
vols = [
    r.get("traffic_volume", 0)
    for r in n1.get("readings", [])
    if isinstance(r, dict)
]
vol_colors = ["#ef4444" if v < 0 else "#3b82f6" for v in vols]

axes[0, 0].bar(range(len(vols)), vols, color=vol_colors)
axes[0, 0].axhline(0, color="black", linewidth=1)
axes[0, 0].set_title("NODE-1 · traffic_volume")
axes[0, 0].set_ylabel("Coches / hora")

# NODE-2: congestion_score
axes[0, 1].bar(range(len(scores)), scores, color=score_colors)
axes[0, 1].axhline(0, color="black", linewidth=1)
axes[0, 1].axhline(0.3, color="green", linestyle="--", alpha=0.5, label="free")
axes[0, 1].axhline(0.6, color="orange", linestyle="--", alpha=0.5, label="moderate")
axes[0, 1].set_title("NODE-2 · congestion_score")
axes[0, 1].set_ylabel("Score")
axes[0, 1].legend(fontsize=8)

# NODE-3: predicciones
from collections import Counter

pred_states = [
    p.get("state", "unknown")
    for p in predictions
    if isinstance(p, dict)
]

if not pred_states:
    pred_states = [
        f.get("label_hint", "unknown")
        for f in features
        if isinstance(f, dict)
    ]

state_counts = Counter(pred_states)
labels, counts = zip(*state_counts.most_common()) if state_counts else (["none"], [0])

bar_colors = {
    "free": "#10b981",
    "moderate": "#f59e0b",
    "heavy": "#ef4444",
    "gridlock": "#7f1d1d",
    "unknown": "#6b7280",
}

axes[1, 0].bar(labels, counts, color=[bar_colors.get(l, "#6b7280") for l in labels])
axes[1, 0].set_title("NODE-3 · Estados predichos")
axes[1, 0].set_ylabel("N predicciones")

# NODE-4: drift
drift = float(metrics.get("drift_score", 0) or 0)
threshold = 0.25
drift_color = "#ef4444" if drift >= threshold else "#10b981"

axes[1, 1].bar(["Drift Score"], [drift], color=drift_color)
axes[1, 1].axhline(
    threshold,
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Umbral ({threshold})",
)
axes[1, 1].set_title("NODE-4 · Model Drift Score")
axes[1, 1].set_ylabel("Drift")
axes[1, 1].set_ylim(0, max(drift + 0.1, threshold + 0.1))
axes[1, 1].legend()
axes[1, 1].text(0, drift + 0.005, f"{drift:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/attack_impact.png", dpi=120, bbox_inches="tight")
plt.show()

print()
print("✓ Observa la cadena de propagación:")
print("  1. NODE-1 acepta una lectura imposible.")
print("  2. NODE-2 la transforma en un congestion_score negativo.")
print("  3. NODE-3 genera inferencia/acciones usando features contaminadas.")
print("  4. NODE-4 evalúa drift y decide si activa retraining.")
print()
print("Acciones reales devueltas por NODE-3:")
if actions:
    for action in actions[:10]:
        print("  →", action)
else:
    print("  No hay acciones disponibles en esta ejecución.")


---
## Paso 3 — Defensa Capa 1: Sanity Checks 🛡️

### Implementa validación de restricciones físicas

El backend real tiene una función `_validate()` en `nodes/sensor_data/app.py`.

Aquí la reimplementas para entender exactamente por qué funciona.

**Objetivo:** rechazar lecturas que violan restricciones físicas básicas antes de que entren al pipeline.


In [ ]:
def validate_reading(reading: dict) -> tuple:
    """
    Valida una lectura de sensor contra restricciones físicas.

    Devuelve:
      (is_valid: bool, reason: str)

    Reglas usadas por NODE-1:
      - temp entre 200K y 350K
      - traffic_volume >= 0
      - rain_1h <= 500
    """

    traffic = reading.get("traffic_volume", 0)
    temp = reading.get("temp", 290)
    rain = reading.get("rain_1h", 0)

    # ── TODO: implementa aquí ────────────────────────────────────────────
    # Descomenta y completa:
    #
    # if not (200 <= temp <= 350):
    #     return False, f"Anomalous temperature: {temp}K"
    #
    # if traffic < ???:
    #     return False, "Negative traffic volume"
    #
    # if rain > 500:
    #     return False, f"Impossible rainfall: {rain}mm"
    # ─────────────────────────────────────────────────────────────────────

    return True, "OK"


casos = [
    {"traffic_volume": 4200, "temp": 288.0, "rain_1h": 0},      # ✓ Legítimo
    {"traffic_volume": 0, "temp": 273.0, "rain_1h": 0},         # ✓ Cero coches es posible
    {"traffic_volume": -5000, "temp": 288.0, "rain_1h": 0},     # ✗ Ataque por volumen negativo
    {"traffic_volume": 4200, "temp": 0.0, "rain_1h": 0},        # ✗ Temperatura imposible
    {"traffic_volume": 4200, "temp": 288.0, "rain_1h": 700},    # ✗ Lluvia imposible
]

print("NODE-1 · Sanity Check — Resultados de tu implementación")
print("=" * 72)

for i, r in enumerate(casos):
    valid, reason = validate_reading(r)
    status = "✓ ACEPTADO" if valid else "✗ RECHAZADO"

    print(
        f"  [{i + 1}] "
        f"vol={r['traffic_volume']:>7} "
        f"temp={r['temp']:>6}K "
        f"rain={r['rain_1h']:>6} "
        f"→ {status} ({reason})"
    )

print()
print("Esperado:")
print("  [1] y [2] ACEPTADOS")
print("  [3], [4] y [5] RECHAZADOS")
print()
print("Si [3] sigue ACEPTADO, revisa el mínimo físico de traffic_volume.")


In [ ]:
# ── SOLUCIÓN DE REFERENCIA ───────────────────────────────────────────────────
# Esta versión es equivalente didáctica a la validación del backend.

# def validate_reading_solution(reading):
#     traffic = reading.get("traffic_volume", 0)
#     temp = reading.get("temp", 290)
#     rain = reading.get("rain_1h", 0)
#
#     if not (200 <= temp <= 350):
#         return False, f"Anomalous temperature: {temp}K"
#
#     if traffic < 0:
#         return False, "Negative traffic volume"
#
#     if rain > 500:
#         return False, f"Impossible rainfall: {rain}mm"
#
#     return True, "OK"
#
#
# print("Solución correcta:")
# print("=" * 72)
#
# for i, r in enumerate(casos):
#     valid, reason = validate_reading_solution(r)
#     status = "✓ ACEPTADO" if valid else "✗ RECHAZADO"
#
#     print(
#         f"  [{i + 1}] "
#         f"vol={r['traffic_volume']:>7} "
#         f"temp={r['temp']:>6}K "
#         f"rain={r['rain_1h']:>6} "
#         f"→ {status} ({reason})"
#     )

print("Celda lista. Descomenta la solución para verificar tu implementación.")


---
## Paso 4 — Defensa Capa 2: Detección de Anomalías Estadísticas 🛡️🛡️

### Z-Score sobre `congestion_score`

Los sanity checks detienen valores físicamente imposibles.

Pero un atacante podría usar valores positivos aparentemente válidos, aunque estadísticamente extraños para una hora concreta.

Aquí construimos un detector estadístico simple usando `congestion_score`.


In [ ]:
print("Recolectando baseline histórico desde el pipeline real...")

all_scores = []

for i in range(5):
    try:
        _, _, n2_i, _, _ = call_pipeline()
        feats = n2_i.get("features", [])

        valid_scores = [
            float(f["congestion_score"])
            for f in feats
            if isinstance(f, dict)
            and isinstance(f.get("congestion_score"), (int, float))
            and 0.0 <= float(f["congestion_score"]) <= 1.0
        ]

        all_scores.extend(valid_scores)

    except Exception as exc:
        print(f"  Llamada {i + 1} fallida: {exc}")

MEAN_SCORE = np.mean(all_scores) if all_scores else 0.45
STD_SCORE = np.std(all_scores) if all_scores else 0.20
STD_SCORE = max(STD_SCORE, 0.01)

print(f"  Baseline construido con {len(all_scores)} scores limpios")
print(f"  mean = {MEAN_SCORE:.4f}")
print(f"  std  = {STD_SCORE:.4f}")


In [ ]:
UMBRAL_Z = 3.0


def detectar_anomalia(feature: dict) -> tuple:
    """
    Detecta anomalías usando Z-Score sobre congestion_score.

    Devuelve:
      (es_anomalia: bool, z_score: float, accion: str)
    """

    score = float(feature.get("congestion_score", 0))
    z = abs((score - MEAN_SCORE) / STD_SCORE)

    if z > UMBRAL_Z:
        return True, round(z, 2), "QUARANTINE"

    return False, round(z, 2), "FORWARD"


print(f"NODE-2 · Detección de Anomalías Z-Score (umbral = {UMBRAL_Z}σ)")
print(f"Baseline: mean={MEAN_SCORE:.3f}, std={STD_SCORE:.3f}")
print("=" * 72)

_, _, n2_fresh, _, _ = call_pipeline()
features_fresh = n2_fresh.get("features", [])

quarantine_count = 0

for f in features_fresh:
    if not isinstance(f, dict):
        continue

    es_anom, z, accion = detectar_anomalia(f)

    flag = "🔴" if es_anom else "🟢"
    vol = f.get("traffic_volume", "?")
    score = f.get("congestion_score", "?")

    if es_anom:
        quarantine_count += 1

    print(
        f"  {flag} "
        f"vol={str(vol):>8} "
        f"score={str(score):>8} "
        f"Z={z:>6.2f} "
        f"→ {accion}"
    )

print()
print(f"  {quarantine_count}/{len(features_fresh)} lecturas en QUARANTINE.")
print("  Modifica UMBRAL_Z y re-ejecuta para ver el tradeoff sensibilidad/especificidad.")


In [ ]:
scores_fresh = [
    float(f.get("congestion_score", 0))
    for f in features_fresh
    if isinstance(f, dict)
]

cols = [
    "#ef4444" if detectar_anomalia(f)[0] else "#3b82f6"
    for f in features_fresh
    if isinstance(f, dict)
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    f"NODE-2 · Z-Score Anomaly Detection (umbral={UMBRAL_Z}σ)",
    fontsize=12,
    fontweight="bold",
)

axes[0].bar(range(len(scores_fresh)), scores_fresh, color=cols)
axes[0].axhline(
    MEAN_SCORE,
    color="green",
    linewidth=2,
    label=f"Media histórica ({MEAN_SCORE:.3f})",
)
axes[0].axhline(
    MEAN_SCORE + UMBRAL_Z * STD_SCORE,
    color="orange",
    linestyle="--",
    label=f"+{UMBRAL_Z}σ",
)
axes[0].axhline(
    MEAN_SCORE - UMBRAL_Z * STD_SCORE,
    color="orange",
    linestyle="--",
    label=f"-{UMBRAL_Z}σ",
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Lectura #")
axes[0].set_ylabel("congestion_score")
axes[0].set_title("Scores del batch — rojo = QUARANTINE")
axes[0].legend(fontsize=8)

axes[1].hist(
    all_scores,
    bins=12,
    color="#3b82f6",
    alpha=0.7,
    label="Baseline histórico",
)

axes[1].axvline(
    MEAN_SCORE - UMBRAL_Z * STD_SCORE,
    color="orange",
    linestyle="--",
    label=f"-{UMBRAL_Z}σ",
)

for s in [sc for sc in scores_fresh if sc < 0]:
    axes[1].axvline(
        s,
        color="red",
        linewidth=2,
        label=f"Envenenado ({s:.3f})",
    )

axes[1].set_xlabel("congestion_score")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Baseline vs. valor envenenado")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/anomaly_detection.png", dpi=120, bbox_inches="tight")
plt.show()

print("✓ El score negativo está fuera del rango histórico y Z-Score lo captura.")


---
## Paso 5 — Defensa Capa 3: Monitorización de Drift 🛡️🛡️🛡️

### NODE-4 — ¿cuándo se debe pausar el reentrenamiento?

El backend calcula un `drift_score` a partir del feedback del nodo de inferencia/decisión.

Si el drift supera:

```python
RETRAIN_DRIFT_THRESHOLD = 0.25
```

entonces se activa el flujo de reentrenamiento.


In [ ]:
metrics, n1, n2, n3, n4 = call_pipeline()

drift_score = float(metrics.get("drift_score", 0) or 0)
retrain_triggered = bool(n4.get("retrain_triggered", False))
retrain_ok = n4.get("retrain_error") is None if retrain_triggered else None
store_ok = n4.get("store_error") is None

THRESHOLD = 0.25

print("NODE-4 · Monitor de Drift del Modelo")
print("=" * 72)
print(f"  drift_score actual:        {drift_score:.4f} ({drift_score * 100:.1f}%)")
print(f"  Umbral de reentrenamiento: {THRESHOLD:.2f} ({THRESHOLD * 100:.0f}%)")
print(f"  ¿Store OK?                 {store_ok}")
print(f"  ¿Reentrenamiento activado? {retrain_triggered}")

if retrain_triggered:
    print(f"  ¿Retraining OK?            {retrain_ok}")

print()

if drift_score >= THRESHOLD:
    print("  🔴 DRIFT SUPERA EL UMBRAL")
    print("  → Se activa el flujo de reentrenamiento.")
    print("  → En un sistema endurecido, debería requerir revisión antes de promocionar modelo.")
else:
    print("  🟢 Drift dentro de límites seguros")
    print("  → No se activa reentrenamiento por drift.")

print()
print("→ Responde en la guía: ¿qué mecanismo debe activarse si el drift es demasiado alto?")


In [ ]:
print("Recolectando drift scores a lo largo de varias llamadas al pipeline...")

drift_history = []

for i in range(8):
    try:
        m, _, _, _, _ = call_pipeline()
        d = float(m.get("drift_score", 0) or 0)
        drift_history.append(d)
    except Exception as exc:
        print(f"  Llamada {i + 1} fallida: {exc}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    "NODE-4 · Monitorización de Model Drift",
    fontsize=12,
    fontweight="bold",
)

call_nums = list(range(1, len(drift_history) + 1))
drift_colors = [
    "#ef4444" if d >= THRESHOLD else "#10b981"
    for d in drift_history
]

axes[0].bar(call_nums, drift_history, color=drift_colors)
axes[0].axhline(
    THRESHOLD,
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Umbral ({THRESHOLD})",
)
axes[0].set_xlabel("Llamada al pipeline #")
axes[0].set_ylabel("drift_score")
axes[0].set_title("Drift score por ejecución")
axes[0].set_ylim(0, max(max(drift_history, default=0) + 0.05, THRESHOLD + 0.1))
axes[0].legend()

labels_dash = [
    "Lecturas\nrecibidas",
    "Envenenadas",
    "Features\nanómalas",
    "Predicciones",
]

values_dash = [
    metrics.get("readings_received", 0),
    metrics.get("poisoned_readings", 0),
    metrics.get("anomalous_features", 0),
    metrics.get("predictions_generated", 0),
]

bar_c = ["#3b82f6", "#ef4444", "#f59e0b", "#10b981"]

axes[1].bar(labels_dash, values_dash, color=bar_c)
axes[1].set_title("Resumen del pipeline")
axes[1].set_ylabel("Cantidad")

plt.tight_layout()
plt.savefig("/tmp/drift_monitoring.png", dpi=120, bbox_inches="tight")
plt.show()

halt_count = sum(1 for d in drift_history if d >= THRESHOLD)

print()
print(f"  {halt_count}/{len(drift_history)} ejecuciones superaron el umbral de drift.")

if drift_history:
    print(f"  Drift medio observado: {np.mean(drift_history):.4f}")


---
## ✅ Lab 1 completado — Resumen de defensa en profundidad

Has analizado el pipeline real de CityFlow AI y has trabajado tres capas defensivas:

| Capa | Nodo | Técnica | Qué detiene |
|------|------|---------|-------------|
| 1 | NODE-1 · Sensor Data | **Sanity Checks** | Valores físicamente imposibles como `traffic_volume = -5000` |
| 2 | NODE-2 · Edge Pre-processing | **Z-Score / Anomaly Detection** | Valores estadísticamente anómalos |
| 3 | NODE-4 · Trainer / Retraining | **Drift Monitoring** | Cambios peligrosos en la distribución antes del reentrenamiento |

### Conclusiones clave

- Una defensa individual no es suficiente.
- Los sanity checks son baratos y bloquean errores evidentes.
- La detección estadística cubre ataques más sutiles.
- El drift monitoring protege el ciclo de reentrenamiento.
- El pipeline debe combinar validación, cuarentena, trazabilidad y revisión antes de promocionar modelos.

---
**Siguiente paso:** completa la guía del panel lateral y responde el quiz final.
